# Classical-AI Reasoning Agent

**Task.** Given a knowledge base (*context*) of facts and rules and a query (*statement*), decide whether
the query is **entailed** by the knowledge base and output `True` / `False` (*flag*).

**Pipeline.**

```
context + statement ──► parser ──► first-order-logic AST ──► engine selection ──► True / False
                         │                                        │
     logic (∀ ∃ → ∧ ∨ ¬ ⊕ ↔, ASCII, Prolog)               (A) forward chaining + unification + stratified negation-as-failure
     templated English (RuleTaker, CounterBench)            (closed-world rule bases)
                                                          (B) resolution refutation, set-of-support, best-first clause
                                                              selection, unification, subsumption (open-world FOL)
                                                          (C) structural causal model: forward propagation + abduction
                                                              by exhaustive search (counterfactual questions)
```


Sections: 0 configuration · 1–9 the reasoning system · 10 data · 11 evaluation on the public datasets ·
12 experiments · 13 reasoning traces · 14 `submission.csv`.

## 0. Configuration
Set `TEST_PATH` to the private test file (CSV / JSON / JSONL / Parquet). Expected columns: `id`, `context`, `statement`
(common alternatives such as `premises`, `question`, `conclusion`, `query` are recognised automatically).
If `TEST_PATH` does not exist, `submission.csv` is produced for the public evaluation sets instead.

In [ ]:
import os, sys, time, json, random, collections
TEST_PATH   = 'test.csv'          
OUTPUT_PATH = 'submission.csv'
DATA_DIR    = 'data'             
LABEL_COLUMN = 'flag'             # prediction column
RULETAKER_EVAL = None             
RUN_EXPERIMENTS = True

In [ ]:

import re, ast, time, heapq, itertools, collections

## 1. Representation: terms and formulas
Variables are strings starting with `?`, constants are plain strings, function terms are tuples. Formulas are nested tuples (`atom`, `not`, `and`, `or`, `imp`, `iff`, `xor`, `forall`, `exists`).

In [ ]:

def is_var(t):
    return isinstance(t, str) and t.startswith('?')


def fmt_term(t):
    if isinstance(t, tuple):
        return '%s(%s)' % (t[0], ', '.join(fmt_term(a) for a in t[1:]))
    return t[1:] if is_var(t) else t


def fmt(f):
    op = f[0]
    if op == 'atom':
        return '%s(%s)' % (f[1], ', '.join(fmt_term(a) for a in f[2])) if f[2] else f[1]
    if op == 'not':
        return '¬' + fmt(f[1])
    if op in ('and', 'or'):
        return '(' + (' ∧ ' if op == 'and' else ' ∨ ').join(fmt(g) for g in f[1:]) + ')'
    if op in ('imp', 'iff', 'xor'):
        sym = {'imp': ' → ', 'iff': ' ↔ ', 'xor': ' ⊕ '}[op]
        return '(' + fmt(f[1]) + sym + fmt(f[2]) + ')'
    if op in ('forall', 'exists'):
        return ('∀' if op == 'forall' else '∃') + f[1][1:] + ' ' + fmt(f[2])
    return op

## 2. Logic parser
A tolerant recursive-descent parser. It accepts Unicode (∀ ∃ → ∧ ∨ ¬ ⊕ ↔ ≠), ASCII (`forall x.`, `->`, `&`, `|`, `~`), word operators (`and`, `or`, `not`, `implies`) and Prolog clauses (`h(X) :- b(X), \\+ c(X).`). Precedence: ¬ > ∧ > ∨ > ⊕ > → > ↔ (→ right-associative). It repairs unbalanced parentheses and other annotation slips found in the public data. Free variables are closed universally in premises and existentially in queries.

In [ ]:

_SYM = [
    ('IFF', ['<->', '<=>', '↔', '⟷', '⇔', '≡', '⟺']),
    ('IMP', ['—>', '–>', '->', '=>', '→', '⇒', '⟶', '⊃', '⟹']),
    ('NEQ', ['!=', '≠', '/=']),
    ('AND', ['&&', '/\\', '∧', '&', '^', '⋀']),
    ('OR',  ['||', '\\/', '∨', '|', '⋁']),
    ('XOR', ['⊕', '⊻']),
    ('NOT', ['¬', '~', '!', '-', '﹁']),
    ('ALL', ['∀']),
    ('EX',  ['∃']),
    ('EQ',  ['=']),
    ('LP',  ['(', '[', '{']),
    ('RP',  [')', ']', '}']),
    ('COMMA', [',']),
    ('DOT', ['.', ':']),
]
_WORD_OPS = {'and': 'AND', 'or': 'OR', 'not': 'NOT', 'implies': 'IMP', 'iff': 'IFF',
             'xor': 'XOR', 'forall': 'ALL', 'exists': 'EX', 'exist': 'EX',
             'true': 'TRUE', 'false': 'FALSE'}
_SYM_LIST = sorted(((s, k) for k, ss in _SYM for s in ss), key=lambda p: -len(p[0]))


class ParseError(Exception):
    pass


def tokenize(s):
    toks, i, n = [], 0, len(s)
    while i < n:
        c = s[i]
        if c.isspace():
            i += 1
            continue
        if c.isalnum() or c == '_':
            j = i
            while j < n and (s[j].isalnum() or s[j] in "_'" or
                             (s[j] in '-.' and j + 1 < n and s[j + 1].isalnum() and j > i
                              and s[j - 1].isalnum())):
                j += 1
            w = s[i:j]
            k = _WORD_OPS.get(w.lower()) if w.lower() in _WORD_OPS and w not in ('True', 'False') else None
            if w in ('TRUE', 'FALSE'):
                k = None
            toks.append((k, w) if k else ('ID', w))
            i = j
            continue
        for sym, kind in _SYM_LIST:
            if s.startswith(sym, i):
                toks.append((kind, sym))
                i += len(sym)
                break
        else:
            i += 1         
    return toks


_PREC = {'IFF': 1, 'IMP': 2, 'XOR': 3, 'OR': 4, 'AND': 5}
_FREE_VAR = re.compile(r'^[u-z][0-9]*$')


class FOLParser:
    def __init__(self, text):
        self.toks = tokenize(text)
        self.i = 0
        self.bound = []

    def peek(self, k=0):
        j = self.i + k
        return self.toks[j] if j < len(self.toks) else (None, None)

    def take(self, kind=None):
        t = self.peek()
        if kind and t[0] != kind:
            raise ParseError('expected %s got %s' % (kind, t))
        self.i += 1
        return t

    def parse(self):
        if not self.toks:
            raise ParseError('empty')
        f = self.expr(0)
        if self.i < len(self.toks):
            # tolerate a trailing stray ')' or '.'
            rest = [t[0] for t in self.toks[self.i:]]
            if all(r in ('RP', 'DOT') for r in rest):
                return f
            if rest[0] in ('ID', 'DOT') and self.i > 3:      # e.g. an English remark
                return f
            raise ParseError('trailing tokens %s' % self.toks[self.i:self.i + 4])
        return f

    def expr(self, minp):
        left = self.unary()
        return self.binary_rest(left, minp)

    def binary_rest(self, left, minp):
        while True:
            k = self.peek()[0]
            if self.peek() == ('ID', 'v') and self.peek(1)[0] in ('LP', 'ID', 'NOT', 'ALL', 'EX'):
                self.toks[self.i] = ('OR', 'v')
                k = 'OR'
            p = _PREC.get(k)
            if p is None or p < minp:
                return left
            self.take()
            if k == 'IMP':          # right associative
                right = self.expr(p)
                left = ('imp', left, right)
            elif k == 'IFF':
                right = self.expr(p)
                left = ('iff', left, right)
            else:
                right = self.expr(p + 1)
                op = {'AND': 'and', 'OR': 'or', 'XOR': 'xor'}[k]
                if op == 'xor':
                    left = ('xor', left, right)
                else:
                    left = flat(op, [left, right])

    def unary(self):
        k, v = self.peek()
        if k == 'NOT':
            self.take()
            return ('not', self.unary())
        if k in ('ALL', 'EX'):
            return self.quantified()
        if k == 'LP':
            self.take()
            f = self.expr(0)
            if self.peek()[0] == 'RP':
                self.take()
            return f
        if k == 'TRUE':
            self.take()
            return ('true',)
        if k == 'FALSE':
            self.take()
            return ('false',)
        if k == 'ID':
            return self.atom()
        raise ParseError('unexpected %s' % (self.peek(),))

    def quantified(self):
        k, _ = self.take()
        q = 'forall' if k == 'ALL' else 'exists'
        names = []
        if self.peek()[0] == 'LP':
            body = self.unary()
            fv = sorted(v for v in free_vars(body) if _FREE_VAR.match(v[1:]))
            for v in reversed(fv[:1]):
                body = (q, v, body)
            return body
        while True:
            t = self.peek()
            if t[0] == 'ID':
                names.append(t[1])
                self.take()
                if self.peek()[0] == 'COMMA' and self.peek(1)[0] == 'ID' and self.peek(2)[0] != 'LP':
                    self.take()
                    continue
                if self.peek()[0] == k:          # ∀x ∀y
                    self.take()
                    continue
                break
            raise ParseError('bad quantifier')
        if self.peek()[0] == 'DOT':
            self.take()
        self.bound.append(set(names))
        body = self.unary()
        # wide scope: if the bound variable is used later in this group,
        # the quantifier extends over the following binary operators.
        if self.peek()[0] in _PREC and self._var_used_later(names):
            body = self.binary_rest(body, 0)
        self.bound.pop()
        for n in reversed(names):
            body = (q, '?' + n, body)
        return body

    def _var_used_later(self, names):
        depth = 0
        for t in self.toks[self.i:]:
            if t[0] == 'LP':
                depth += 1
            elif t[0] == 'RP':
                depth -= 1
                if depth < 0:
                    return False
            elif t[0] == 'ID' and t[1] in names:
                return True
        return False

    def is_bound(self, name):
        return any(name in b for b in self.bound)

    def term(self):
        while self.peek()[0] == 'LP':
            self.take()
        if self.peek()[0] in ('TRUE', 'FALSE'):
            return self.take()[1].lower()
        k, v = self.take('ID')
        if self.peek()[0] == 'LP':
            self.take()
            args = self.args()
            return (v,) + tuple(args)
        if self.is_bound(v) or _FREE_VAR.match(v):
            return '?' + v
        return v

    def args(self):
        args = []
        if self.peek()[0] == 'RP':
            self.take()
            return args
        while True:
            args.append(self.term())
            t = self.peek()[0]
            if t == 'COMMA':
                self.take()
                if self.peek()[0] == 'RP':
                    self.take()
                    break
                continue
            if t == 'RP':
                self.take()
            break
        return args

    def atom(self):
        k, name = self.take('ID')
        if self.peek()[0] == 'LP':
            self.take()
            args = tuple(self.args())
            a = ('atom', name, args)
        else:
            # term = term   or propositional atom
            if self.peek()[0] in ('EQ', 'NEQ'):
                self.i -= 1
                left = self.term()
                op = self.take()[0]
                right = self.term()
                a = ('atom', '=', (left, right))
                return a if op == 'EQ' else ('not', a)
            a = ('atom', name, ())
        if self.peek()[0] in ('EQ', 'NEQ'):           # f(x) = c
            op = self.take()[0]
            right = self.term()
            left = (name,) + tuple(a[2]) if a[2] else name
            a = ('atom', '=', (left, right))
            return a if op == 'EQ' else ('not', a)
        return a


def flat(op, parts):
    out = []
    for p in parts:
        if p[0] == op:
            out.extend(p[1:])
        else:
            out.append(p)
    return (op,) + tuple(out)


def _balance(s):
    depth, low = 0, 0
    for ch in s:
        if ch == '(':
            depth += 1
        elif ch == ')':
            depth -= 1
            low = min(low, depth)
    s = '(' * (-low) + s
    depth -= low
    return s + ')' * depth


def parse_prolog(line):
    """Prolog clause  head :- b1, b2.   (uppercase identifiers are variables)"""
    line = line.strip().rstrip('.')
    def conv(t):
        t = t.strip()
        neg = False
        if t.startswith('\\+') or t.startswith('not '):
            neg = True
            t = t[2:] if t.startswith('\\+') else t[4:]
        t = re.sub(r'\b([A-Z_][A-Za-z0-9_]*)\b(?!\s*\()', lambda m: 'V_' + m.group(1), t)
        return ('¬' if neg else '') + t
    if ':-' in line:
        head, body = line.split(':-', 1)
        parts = re.split(r',(?![^()]*\))', body)
        txt = '(' + ' ∧ '.join(conv(p) for p in parts) + ') → ' + conv(head)
    else:
        txt = conv(line)
    f = FOLParser(txt).parse()
    f = rename_prolog_vars(f)
    return close_free(f)


def rename_prolog_vars(f):
    def rt(t):
        if isinstance(t, tuple):
            return (t[0],) + tuple(rt(a) for a in t[1:])
        if isinstance(t, str) and t.startswith('V_'):
            return '?' + t[2:]
        return t
    return map_atoms(f, lambda a: ('atom', a[1], tuple(rt(x) for x in a[2])))


def map_atoms(f, fn):
    op = f[0]
    if op == 'atom':
        return fn(f)
    if op == 'not':
        return ('not', map_atoms(f[1], fn))
    if op in ('and', 'or'):
        return (op,) + tuple(map_atoms(g, fn) for g in f[1:])
    if op in ('imp', 'iff', 'xor'):
        return (op, map_atoms(f[1], fn), map_atoms(f[2], fn))
    if op in ('forall', 'exists'):
        return (op, f[1], map_atoms(f[2], fn))
    return f


def term_vars(t, acc):
    if isinstance(t, tuple):
        for a in t[1:]:
            term_vars(a, acc)
    elif is_var(t):
        acc.add(t)


def free_vars(f, bound=frozenset()):
    op = f[0]
    if op == 'atom':
        acc = set()
        for a in f[2]:
            term_vars(a, acc)
        return acc - bound
    if op == 'not':
        return free_vars(f[1], bound)
    if op in ('and', 'or', 'imp', 'iff', 'xor'):
        r = set()
        for g in f[1:]:
            r |= free_vars(g, bound)
        return r
    if op in ('forall', 'exists'):
        return free_vars(f[2], bound | {f[1]})
    return set()


def close_free(f, q='forall'):
    for v in sorted(free_vars(f)):
        f = (q, v, f)
    return f


def parse_logic(text, free='forall'):
    text = text.strip().rstrip(';')
    if not text:
        raise ParseError('empty')
    if ':-' in text or re.match(r'^[a-z_][\w]*\(.*\)\s*\.$', text):
        return parse_prolog(text)
    text = text.rstrip('.')
    try:
        f = FOLParser(text).parse()
    except ParseError:
        f = FOLParser(_balance(text)).parse()
    return close_free(f, free)


_LOGIC_HINT = re.compile(r'[∀∃→∧∨¬⊕↔⟷⇒⇔]|->|<->|=>|&|\||:-|~\s*\w|\b(forall|exists)\b|^\s*!?\w+\(|^\s*\w+\s*!?=\s*\w+\s*$')


def looks_like_logic(s):
    if _LOGIC_HINT.search(s):
        return True
    return bool(re.search(r'\b[A-Za-z_]\w*\s*\([^()]*\)', s)) and not re.search(r'\b(is|are|the|if|then)\b', s)

## 3. Templated-English grammar parser


In [ ]:

_VAR_SUBJ = {'someone', 'something', 'they', 'it', 'them', 'he', 'she', 'everyone',
             'everything', 'somebody', 'person', 'thing', 'people', 'things'}


def _ent(e):
    e = e.strip().lower()
    e = re.sub(r'^(the|a|an)\s+', '', e)
    if e in _VAR_SUBJ:
        return '?x'
    return re.sub(r'\s+', '_', e)


def _verb_lemma(v):
    v = v.lower()
    if v.endswith('ies'):
        return v[:-3] + 'y'
    if v.endswith(('ches', 'shes', 'sses', 'xes', 'zes')):
        return v[:-2]
    if v.endswith('s') and not v.endswith('ss'):
        return v[:-1]
    return v


_ENT = r"(?:[Tt]he\s+[a-z]+|[Tt]he\s+[a-z]+\s+[a-z]+|[A-Z][a-z]+|someone|something|they|it|them|he|she)"
_ADJ = r"[a-z]+"


def _lit(pred, args, neg):
    a = ('atom', pred, tuple(args))
    return ('not', a) if neg else a


def parse_clause(c, default_subj=None):
    """Parse a simple clause into a list of literals (conjunction)."""
    c = c.strip().rstrip('.').strip()
    # "<E> is/are [not] A [and [not] B ...]"
    m = re.match(r'^(%s)\s+(?:is|are)\s+(.*)$' % _ENT, c)
    if m:
        subj = _ent(m.group(1))
        rest = m.group(2)
        parts = re.split(r'\s*(?:,\s*and|,|\band\b)\s*', rest)
        lits = []
        for p in parts:
            p = p.strip()
            if not p:
                continue
            mm = re.match(r'^(not\s+)?(?:a\s+|an\s+)?(%s)$' % _ADJ, p)
            if not mm:
                sub = parse_clause(p)
                if sub is None:
                    return None
                lits.extend(sub)
                continue
            lits.append(_lit(mm.group(2).lower(), [subj], bool(mm.group(1))))
        return lits
    # "<E> does not / do not <verb> <E2>"
    m = re.match(r'^(%s)\s+(?:does|do)\s+not\s+([a-z]+)\s+(%s)$' % (_ENT, _ENT), c)
    if m:
        return [_lit(_verb_lemma(m.group(2)), [_ent(m.group(1)), _ent(m.group(3))], True)]
    # "<E> <verb>s <E2>"
    m = re.match(r'^(%s)\s+([a-z]+)\s+(%s)$' % (_ENT, _ENT), c)
    if m and m.group(2).lower() not in ('is', 'are'):
        return [_lit(_verb_lemma(m.group(2)), [_ent(m.group(1)), _ent(m.group(3))], False)]
    return None


def _conj(lits):
    return lits[0] if len(lits) == 1 else ('and',) + tuple(lits)


def _rule(body, head):
    f = ('imp', _conj(body), _conj(head))
    return close_free(f)


def parse_english_sentence(s):
    s = s.strip().rstrip('.').strip()
    if not s:
        return None
    # If ... then ...
    m = re.match(r'^if\s+(.*?)\s*,?\s+then\s+(.*)$', s, re.I)
    if m:
        conds = _split_conds(m.group(1))
        body = []
        for c in conds:
            l = parse_clause(c)
            if l is None:
                return None
            body.extend(l)
        head = parse_clause(m.group(2))
        if head is None:
            return None
        return _rule(body, head)
    # All A, B people/things are [not] C.     A, B things are C.
    m = re.match(r'^(?:all\s+)?([a-z][a-z ,]*?)\s+(people|things|persons|animals|ones)\s+are\s+(.*)$', s, re.I)
    if m:
        adjs = [a for a in re.split(r'\s*,\s*|\s+and\s+|\s+', m.group(1).strip().lower()) if a and a != 'all']
        body = [_lit(a, ['?x'], False) for a in adjs]
        head = parse_clause('it is ' + m.group(3))
        if head is None or not body:
            return None
        return _rule(body, head)
    # Someone/Something that is A is B  |  A people are B
    lits = parse_clause(s)
    if lits is None:
        return None
    return _conj(lits) if len(lits) > 1 else lits[0]


def _split_conds(txt):
    # split "someone is big and they chase the cat" into clauses, but keep
    # "someone is big and red" together (an adjective after 'and').
    parts = re.split(r'\s+and\s+', txt)
    out = []
    for p in parts:
        if out and (re.match(r'^(not\s+)?[a-z]+$', p) and p.split()[-1] not in _VAR_SUBJ):
            out[-1] = out[-1] + ' and ' + p
        else:
            out.append(p)
    return out


def split_sentences(text):
    return [s for s in re.split(r'(?<=[.!?])\s+|\n+', text.strip()) if s.strip()]

## 4. Front-end: knowledge base and query to formulas
Splits the context into premises (newlines, sentences, or several formulas on one line) and routes each to the logic parser or the English grammar.

In [ ]:
def _premise_lines(context):
    if isinstance(context, (list, tuple)):
        return [str(x) for x in context]
    c = str(context).strip()
    if c.startswith('[') and c.endswith(']'):
        try:
            v = ast.literal_eval(c)
            if isinstance(v, list):
                return [str(x) for x in v]
        except Exception:
            pass
    lines = [l for l in re.split(r'\n+', c) if l.strip()]
    return lines


_QUANT_TAIL = re.compile(r'(?:[∀∃]|\b(?:forall|exists|all|some)\b)\s*\w+(?:\s*,\s*\w+)*\s*$', re.I)


def split_logic_line(s):
    parts, depth, start = [], 0, 0
    for i, ch in enumerate(s):
        if ch in '([{':
            depth += 1
        elif ch in ')]}':
            depth = max(0, depth - 1)        
        elif depth == 0 and ch in ';,.':
            nxt = s[i + 1:i + 2]
            prev = s[start:i]
            if ch == '.' and nxt and not nxt.isspace():
                continue                    
            if _QUANT_TAIL.search(prev):
                continue                      # ∀x.  /  ∀x, y
            if ch == ',' and not re.search(r'[)\w]\s*$', prev):
                continue
            if prev.strip():
                parts.append(prev)
            start = i + 1
    if s[start:].strip():
        parts.append(s[start:])
    return parts


def parse_statement(s, stats=None, free='forall'):
    """Return a list of formulas for one input line (logic or English)."""
    s = s.strip()
    if not s:
        return []
    if looks_like_logic(s):
        pieces = split_logic_line(s)
        if len(pieces) <= 1:
            pieces = [s]
        out = []
        for piece in pieces:
            try:
                out.append(parse_logic(piece, free))
            except ParseError:
                if stats is not None:
                    stats['logic_fail'] += 1
        if not out and len(pieces) > 1:
            try:
                out.append(parse_logic(s, free))
            except ParseError:
                pass
        return out
    out = []
    for sent in split_sentences(s):
        f = parse_english_sentence(sent)
        if f is None:
            if stats is not None:
                stats['english_fail'] += 1
                stats.setdefault('examples', []).append(sent)
            continue
        out.append(f)
    return out


def parse_kb(context, stats=None, free='forall'):
    kb = []
    for line in _premise_lines(context):
        kb.extend(parse_statement(line, stats, free))
    return kb

## 5. Engine A: forward chaining with stratified negation-as-failure
For rule bases made of ground literals and rules whose bodies/heads are literals. Semi-naive forward chaining with unification (pattern matching against the fact base); negated heads produce explicit negative facts, negated body literals are evaluated as **negation-as-failure** stratum by stratum (stratification of the predicate dependency graph). Queries are answered under the **closed-world assumption**, the semantics of RuleTaker.

In [ ]:

def _strip_forall(f):
    vs = []
    while f[0] == 'forall':
        vs.append(f[1])
        f = f[2]
    return f, vs


def _literal(f):
    """('atom'...) or ('not', atom)  ->  (positive, pred, args)  or None"""
    if f[0] == 'atom':
        return (True, f[1], f[2])
    if f[0] == 'not' and f[1][0] == 'atom':
        return (False, f[1][1], f[1][2])
    if f[0] == 'not' and f[1][0] == 'not':
        return _literal(f[1][1])
    return None


def _conj_literals(f):
    parts = f[1:] if f[0] == 'and' else (f,)
    out = []
    for p in parts:
        if p[0] == 'and':
            sub = _conj_literals(p)
            if sub is None:
                return None
            out.extend(sub)
            continue
        l = _literal(p)
        if l is None:
            return None
        out.append(l)
    return out


def _has_func(args):
    return any(isinstance(a, tuple) for a in args)


def to_rules(kb):
    facts, rules = [], []
    for f in kb:
        body_f, vs = _strip_forall(f)
        if body_f[0] == 'imp':
            body = _conj_literals(body_f[1])
            head = _conj_literals(body_f[2])
            if body is None or head is None:
                return None
            for h in head:
                if _has_func(h[2]):
                    return None
                rules.append((tuple(body), h))
            continue
        lits = _conj_literals(body_f)
        if lits is None:
            return None
        for l in lits:
            vs_ = set()
            for a in l[2]:
                term_vars(a, vs_)
            if vs_:
                # universally-quantified literal  ∀x P(x)  ->  rule with empty body
                rules.append(((), l))
            else:
                facts.append(l)
    return facts, rules


def _match(pattern_args, fact_args, s):
    s = dict(s)
    for p, a in zip(pattern_args, fact_args):
        if is_var(p):
            if p in s:
                if s[p] != a:
                    return None
            else:
                s[p] = a
        elif p != a:
            return None
    return s


def _subst_args(args, s):
    return tuple(s.get(a, a) if is_var(a) else a for a in args)


class ForwardChainer:
    
    def __init__(self, facts, rules):
        self.rules = rules
        self.trace = {}
        self.consts = set()
        for pos, p, args in facts:
            self.consts.update(args)
        for body, head in rules:
            for l in list(body) + [head]:
                self.consts.update(a for a in l[2] if not is_var(a))
        self.db = collections.defaultdict(set)      # key -> set(args)
        for pos, p, args in facts:
            self.db[self._key(pos, p)].add(args)
            self.trace.setdefault((self._key(pos, p), args), 'given')
        self._run()

    @staticmethod
    def _key(pos, p):
        return p if pos else '¬' + p

    def _strata(self):
        preds = set()
        for body, head in self.rules:
            preds.add(self._key(head[0], head[1]))
            for l in body:
                preds.add(self._key(l[0], l[1]))
        level = {p: 0 for p in preds}
        n = len(preds) + 1
        changed = True
        for _ in range(n + 1):
            changed = False
            for body, head in self.rules:
                h = self._key(head[0], head[1])
                for l in body:
                    if l[0]:
                        need = level[self._key(True, l[1])]
                    else:
                        need = level.get(l[1], 0) + 1      # NAF on the positive predicate
                    if level[h] < need:
                        level[h] = need
                        changed = True
            if not changed:
                break
        if changed:                               
            return None
        return level

    def _run(self):
        level = self._strata()
        if level is None:                         
            groups = [self.rules]
        else:
            by = collections.defaultdict(list)
            for r in self.rules:
                by[level[self._key(r[1][0], r[1][1])]].append(r)
            groups = [by[k] for k in sorted(by)]
        for g in groups:
            self._fixpoint(g)

    def _solve_body(self, body, s, i):
        if i == len(body):
            yield s
            return
        pos, p, args = body[i]
        if pos:
            for fa in list(self.db.get(p, ())):
                s2 = _match(args, fa, s)
                if s2 is not None:
                    yield from self._solve_body(body, s2, i + 1)
        else:
            # NAF: bind remaining variables over the Herbrand universe
            free = [a for a in args if is_var(a) and a not in s]
            if free:
                for vals in itertools.product(sorted(self.consts), repeat=len(set(free))):
                    s2 = dict(s)
                    s2.update(zip(sorted(set(free)), vals))
                    if _subst_args(args, s2) not in self.db.get(p, ()):
                        yield from self._solve_body(body, s2, i + 1)
            else:
                if _subst_args(args, s) not in self.db.get(p, ()):
                    yield from self._solve_body(body, s, i + 1)

    def _fixpoint(self, rules):
        # positive literals first so NAF literals are checked with bound vars
        rules = [(tuple(sorted(b, key=lambda l: not l[0])), h) for b, h in rules]
        changed = True
        while changed:
            changed = False
            for body, head in rules:
                for s in list(self._solve_body(body, {}, 0)):
                    hargs = _subst_args(head[2], s)
                    if any(is_var(a) for a in hargs):
                        free = sorted({a for a in hargs if is_var(a)})
                        insts = [_subst_args(hargs, dict(zip(free, v)))
                                 for v in itertools.product(sorted(self.consts), repeat=len(free))]
                    else:
                        insts = [hargs]
                    k = self._key(head[0], head[1])
                    for ha in insts:
                        if ha not in self.db[k]:
                            self.db[k].add(ha)
                            self.trace[(k, ha)] = (body, head, s)
                            changed = True

    def holds(self, pos, p, args):
        if pos:
            return args in self.db.get(p, ())
        return args in self.db.get('¬' + p, ()) or args not in self.db.get(p, ())

    def explain(self, pos, p, args, depth=0, out=None):
        out = [] if out is None else out
        k = self._key(pos, p)
        t = self.trace.get((k, args))
        lit = ('' if pos else '¬') + '%s(%s)' % (p, ', '.join(args))
        if t is None:
            out.append('  ' * depth + lit + '   [closed-world: not derivable]')
        elif t == 'given':
            out.append('  ' * depth + lit + '   [fact]')
        else:
            body, head, s = t
            out.append('  ' * depth + lit + '   [rule: %s → %s]' % (
                ' ∧ '.join(('' if b[0] else 'not ') + '%s(%s)' % (b[1], ', '.join(fmt_term(a) for a in b[2])) for b in body),
                ('' if head[0] else '¬') + '%s(%s)' % (head[1], ', '.join(fmt_term(a) for a in head[2]))))
            for b in body:
                if depth < 12:
                    self.explain(b[0], b[1], _subst_args(b[2], s), depth + 1, out)
        return out

## 6. Engine B: resolution refutation for first-order logic
Formulas are converted to clausal form (→/↔/⊕ elimination, negation normal form, standardising apart, Skolemisation, distribution). The negated query forms the **set of support**; the given-clause loop always picks the lightest clause (a **uniform-cost / best-first search** in the space of derived clauses), resolves it against complementary literals via an index, applies factoring, tautology deletion and θ-subsumption, and stops at the empty clause □. Equality is handled with equality axioms plus the unique-names assumption.

In [ ]:
# -----------------------------------------------------------------------------
# 6. Engine B: resolution refutation for full first-order logic
# -----------------------------------------------------------------------------
class _Fresh:
    n = 0

    @classmethod
    def var(cls):
        cls.n += 1
        return '?v%d' % cls.n

    @classmethod
    def skolem(cls):
        cls.n += 1
        return 'sk%d' % cls.n


def eliminate(f):
    """Remove →, ↔, ⊕ (keeps quantifiers)."""
    op = f[0]
    if op == 'atom' or op in ('true', 'false'):
        return f
    if op == 'not':
        return ('not', eliminate(f[1]))
    if op in ('and', 'or'):
        return (op,) + tuple(eliminate(g) for g in f[1:])
    if op in ('forall', 'exists'):
        return (op, f[1], eliminate(f[2]))
    a, b = eliminate(f[1]), eliminate(f[2])
    if op == 'imp':
        return ('or', ('not', a), b)
    if op == 'iff':
        return ('and', ('or', ('not', a), b), ('or', a, ('not', b)))
    if op == 'xor':
        return ('and', ('or', a, b), ('or', ('not', a), ('not', b)))
    raise ValueError(op)


def nnf(f, neg=False):
    op = f[0]
    if op == 'atom':
        return ('not', f) if neg else f
    if op == 'true':
        return ('false',) if neg else f
    if op == 'false':
        return ('true',) if neg else f
    if op == 'not':
        return nnf(f[1], not neg)
    if op in ('and', 'or'):
        nop = op if not neg else ('or' if op == 'and' else 'and')
        return flat(nop, [nnf(g, neg) for g in f[1:]])
    if op in ('forall', 'exists'):
        nop = op if not neg else ('exists' if op == 'forall' else 'forall')
        return (nop, f[1], nnf(f[2], neg))
    raise ValueError(op)


def _subst_term(t, s):
    if isinstance(t, tuple):
        return (t[0],) + tuple(_subst_term(a, s) for a in t[1:])
    return s.get(t, t) if is_var(t) else t


def skolemize(f, univ=(), ren=None):
    """Standardise apart, Skolemise existentials, drop universals (NNF input)."""
    ren = ren or {}
    op = f[0]
    if op == 'atom':
        return ('atom', f[1], tuple(_subst_term(a, ren) for a in f[2]))
    if op == 'not':
        return ('not', skolemize(f[1], univ, ren))
    if op in ('and', 'or'):
        return (op,) + tuple(skolemize(g, univ, ren) for g in f[1:])
    if op == 'forall':
        v = _Fresh.var()
        r2 = dict(ren)
        r2[f[1]] = v
        return skolemize(f[2], univ + (v,), r2)
    if op == 'exists':
        sk = _Fresh.skolem()
        r2 = dict(ren)
        r2[f[1]] = (sk,) + tuple(univ) if univ else sk
        return skolemize(f[2], univ, r2)
    return f


def cnf_clauses(f, limit=4000):
    """NNF+skolemised formula without quantifiers -> list of clauses
    (each clause is a list of literals (sign, pred, args))."""
    op = f[0]
    if op == 'atom':
        return [[(True, f[1], f[2])]]
    if op == 'not':
        a = f[1]
        return [[(False, a[1], a[2])]]
    if op == 'true':
        return []
    if op == 'false':
        return [[]]
    if op == 'and':
        out = []
        for g in f[1:]:
            out.extend(cnf_clauses(g, limit))
        return out
    if op == 'or':
        acc = [[]]
        for g in f[1:]:
            cg = cnf_clauses(g, limit)
            acc = [a + b for a in acc for b in cg]
            if len(acc) > limit:
                raise OverflowError
        return acc
    raise ValueError(op)


def to_clauses(f):
    g = skolemize(nnf(eliminate(f)))
    out = []
    for c in cnf_clauses(g):
        c = _simplify_clause(c)
        if c is not None:
            out.append(c)
    return out


def _simplify_clause(c):
    s = set()
    for l in c:
        if (not l[0], l[1], l[2]) in s:
            return None                        # tautology
        s.add(l)
    return tuple(sorted(s, key=repr))


# --- unification -------------------------------------------------------------
def _walk(t, s):
    while is_var(t) and t in s:
        t = s[t]
    return t


def _occurs(v, t, s):
    t = _walk(t, s)
    if t == v:
        return True
    if isinstance(t, tuple):
        return any(_occurs(v, a, s) for a in t[1:])
    return False


def unify(a, b, s):
    a, b = _walk(a, s), _walk(b, s)
    if a == b:
        return s
    if is_var(a):
        if _occurs(a, b, s):
            return None
        s = dict(s)
        s[a] = b
        return s
    if is_var(b):
        return unify(b, a, s)
    if isinstance(a, tuple) and isinstance(b, tuple) and a[0] == b[0] and len(a) == len(b):
        for x, y in zip(a[1:], b[1:]):
            s = unify(x, y, s)
            if s is None:
                return None
        return s
    return None


def unify_args(xs, ys, s):
    if len(xs) != len(ys):
        return None
    for x, y in zip(xs, ys):
        s = unify(x, y, s)
        if s is None:
            return None
    return s


def _resolve_term(t, s):
    t = _walk(t, s)
    if isinstance(t, tuple):
        return (t[0],) + tuple(_resolve_term(a, s) for a in t[1:])
    return t


def _apply(lit, s):
    return (lit[0], lit[1], tuple(_resolve_term(a, s) for a in lit[2]))


def _rename(clause, tag):
    m = {}
    def r(t):
        if isinstance(t, tuple):
            return (t[0],) + tuple(r(a) for a in t[1:])
        if is_var(t):
            if t not in m:
                m[t] = '?%s%d' % (tag, len(m))
            return m[t]
        return t
    return tuple((l[0], l[1], tuple(r(a) for a in l[2])) for l in clause)


def _normalize(clause):
    c = _simplify_clause(clause)
    if c is None:
        return None
    c = _rename(c, 'n')
    return tuple(sorted(set(c), key=repr))


def _term_size(t):
    if isinstance(t, tuple):
        return 1 + sum(_term_size(a) for a in t[1:])
    return 1


def _weight(c):
    return sum(1 + sum(_term_size(a) for a in l[2]) for l in c) + 2 * len(c)


def _max_depth(t):
    if isinstance(t, tuple):
        return 1 + max((_max_depth(a) for a in t[1:]), default=0)
    return 0


def _subsumes(d, c):
    """θ-subsumption: does clause d subsume clause c ?"""
    if len(d) > len(c):
        return False
    def bt(i, s):
        if i == len(d):
            return True
        l = d[i]
        for m in c:
            if m[0] == l[0] and m[1] == l[1] and len(m[2]) == len(l[2]):
                s2 = _match_terms(l[2], m[2], s)
                if s2 is not None and bt(i + 1, s2):
                    return True
        return False
    return bt(0, {})


def _match_terms(ps, ts, s):
    for p, t in zip(ps, ts):
        s = _match_term(p, t, s)
        if s is None:
            return None
    return s


def _match_term(p, t, s):
    if is_var(p):
        if p in s:
            return s if s[p] == t else None
        s = dict(s)
        s[p] = t
        return s
    if isinstance(p, tuple):
        if isinstance(t, tuple) and p[0] == t[0] and len(p) == len(t):
            return _match_terms(p[1:], t[1:], s)
        return None
    return s if p == t else None


def equality_axioms(clauses):
    preds, funcs = set(), set()
    def ft(t):
        if isinstance(t, tuple):
            funcs.add((t[0], len(t) - 1))
            for a in t[1:]:
                ft(a)
    uses_eq = False
    for c in clauses:
        for l in c:
            if l[1] == '=':
                uses_eq = True
            else:
                preds.add((l[1], len(l[2])))
            for a in l[2]:
                ft(a)
    if not uses_eq:
        return []
    # unique-names assumption: distinct constants denote distinct objects,
    # unless the premises explicitly equate them
    consts = set()
    def cc(t):
        if isinstance(t, tuple):
            for a in t[1:]:
                cc(a)
        elif not is_var(t) and not re.match(r'^sk\d+$', t):
            consts.add(t)
    for c in clauses:
        for l in c:
            for a in l[2]:
                cc(a)
    parent = {c: c for c in consts}
    def find(c):
        while parent[c] != c:
            c = parent[c]
        return c
    for c in clauses:
        if len(c) == 1 and c[0][0] and c[0][1] == '=':
            a, b = c[0][2]
            if a in parent and b in parent:
                parent[find(a)] = find(b)
    una = [((False, '=', (a, b)),) for a, b in itertools.combinations(sorted(consts), 2)
           if find(a) != find(b)] if len(consts) <= 40 else []
    ax = una + [((True, '=', ('?x', '?x')),),
          ((False, '=', ('?x', '?y')), (True, '=', ('?y', '?x'))),
          ((False, '=', ('?x', '?y')), (False, '=', ('?y', '?z')), (True, '=', ('?x', '?z')))]
    for p, n in preds:
        for i in range(n):
            xs = tuple('?a%d' % j for j in range(n))
            ys = tuple('?b' if j == i else '?a%d' % j for j in range(n))
            ax.append(((False, '=', ('?a%d' % i, '?b')), (False, p, xs), (True, p, ys)))
    for fn, n in funcs:
        for i in range(n):
            xs = (fn,) + tuple('?a%d' % j for j in range(n))
            ys = (fn,) + tuple('?b' if j == i else '?a%d' % j for j in range(n))
            ax.append(((False, '=', ('?a%d' % i, '?b')), (True, '=', (xs, ys))))
    return ax


class ResolutionProver:
    """Given-clause resolution (Otter loop) with set-of-support.
    Clause selection is best-first by clause weight (a uniform-cost search
    over the space of derived clauses)."""

    def __init__(self, max_clauses=6000, time_limit=2.0, max_term_depth=3, max_lits=6):
        self.max_clauses = max_clauses
        self.time_limit = time_limit
        self.max_term_depth = max_term_depth
        self.max_lits = max_lits

    def prove(self, kb_clauses, goal_clauses):
        """True iff kb_clauses ∪ goal_clauses is found unsatisfiable."""
        t0 = time.perf_counter()
        usable = []
        index = collections.defaultdict(list)      # (sign, pred) -> [(clause, lit)]
        seen = set()
        self.proof = None
        parents = {}

        def add_usable(c):
            usable.append(c)
            for l in c:
                index[(l[0], l[1])].append(c)

        for c in kb_clauses:
            n = _normalize(c)
            if n is not None and n not in seen:
                seen.add(n)
                add_usable(n)
                parents[n] = 'premise'
        sos, counter = [], itertools.count()
        for c in goal_clauses:
            n = _normalize(c)
            if n is None:
                continue
            if not n:
                return True
            if n not in seen:
                seen.add(n)
                parents[n] = 'negated goal'
                heapq.heappush(sos, (_weight(n), next(counter), n))
        units = [c for c in usable if len(c) == 1]
        generated = 0
        while sos:
            if time.perf_counter() - t0 > self.time_limit or generated > self.max_clauses:
                return False
            _, _, g = heapq.heappop(sos)
            if any(_subsumes(u, g) for u in units if u is not g):
                continue
            gr = _rename(g, 'g')
            partners = {}                          # ordered, deterministic
            for l in gr:
                for c in index.get((not l[0], l[1]), ()):
                    partners[c] = True
            new = []
            for c in list(partners) + [g]:
                cr = _rename(c, 'c')
                for i, l1 in enumerate(gr):
                    for j, l2 in enumerate(cr):
                        if l1[0] == l2[0] or l1[1] != l2[1]:
                            continue
                        s = unify_args(l1[2], l2[2], {})
                        if s is None:
                            continue
                        res = [_apply(x, s) for k, x in enumerate(gr) if k != i] + \
                              [_apply(x, s) for k, x in enumerate(cr) if k != j]
                        new.append((res, g, c))
            # factoring
            for i in range(len(gr)):
                for j in range(i + 1, len(gr)):
                    a, b = gr[i], gr[j]
                    if a[0] == b[0] and a[1] == b[1]:
                        s = unify_args(a[2], b[2], {})
                        if s is not None:
                            new.append(([_apply(x, s) for k, x in enumerate(gr) if k != j], g, g))
            add_usable(g)
            if len(g) == 1:
                units.append(g)
            for res, p1, p2 in new:
                n = _normalize(res)
                if n is None:
                    continue
                if not n:
                    parents[()] = (p1, p2)
                    self.proof = (parents, ())
                    return True
                if n in seen or len(n) > self.max_lits:
                    continue
                if any(_max_depth(a) > self.max_term_depth for l in n for a in l[2]):
                    continue
                if any(_subsumes(u, n) for u in units):
                    continue
                seen.add(n)
                parents[n] = (p1, p2)
                generated += 1
                heapq.heappush(sos, (_weight(n), next(counter), n))
        return False

    def proof_lines(self):
        if not self.proof:
            return []
        parents, root = self.proof
        lines, done = [], {}
        def show(c):
            return '□' if not c else ' ∨ '.join(('' if l[0] else '¬') + '%s(%s)' % (l[1], ', '.join(fmt_term(a) for a in l[2])) for l in c)
        def visit(c, depth=0):
            if c in done or depth > 40:
                return done.get(c, '?')
            p = parents.get(c)
            if p is None or isinstance(p, str):
                done[c] = len(done) + 1
                lines.append('%3d. %-60s [%s]' % (done[c], show(c), p or 'premise'))
                return done[c]
            a = visit(p[0], depth + 1)
            b = visit(p[1], depth + 1)
            done[c] = len(done) + 1
            lines.append('%3d. %-60s [resolve %s, %s]' % (done[c], show(c), a, b))
            return done[c]
        visit(root)
        return lines

## 7. Engine C: structural causal models for counterfactual queries
Each "X causes Y" sentence becomes a Boolean structural equation. Counterfactuals follow abduction–action–prediction: **enumerate** all exogenous assignments (exhaustive DFS over a binary tree), keep those consistent with the observations, apply the intervention do(·), **forward-propagate** the equations in topological order and return P(target) > 0.5.

In [ ]:
# -----------------------------------------------------------------------------
# 7. Engine C: structural causal models for counterfactual questions
# -----------------------------------------------------------------------------
_CF_Q = re.compile(r'\bwould\b.*\boccur\b', re.I)


def is_counterfactual(statement):
    return bool(_CF_Q.search(statement))


def parse_scm(text):
    body = re.sub(r'(?i)^.*?\bwe know that\s+', '', text.strip())
    priors, obs, eqs = {}, {}, {}
    for v, p in re.findall(r'([A-Z]\w*)\s*~\s*Bern\(\s*([\d.]+)\s*\)', body):
        priors[v] = float(p)
    body = re.sub(r'[A-Z]\w*\s*~\s*Bern\(\s*[\d.]+\s*\)\.?', '', body)
    mo = re.search(r'We observed\s+(.*)', body, re.I | re.S)
    if mo:
        for part in re.split(r',|\band\b', mo.group(1)):
            mm = re.search(r'(not\s+)?\b([A-Z]\w*)\b(.*)', part)
            if mm:
                neg = bool(mm.group(1)) or bool(re.search(r"\bnot\b|n't", mm.group(3)))
                obs[mm.group(2)] = 0 if neg else 1
        body = body[:mo.start()]
    verb = r'\s+(?:together\s+)?causes?\s+'
    def lits_of(txt):
        out = []
        for p in re.split(r'\band\b|\bor\b', txt):
            p = p.strip()
            if p:
                pn = p.startswith('not ')
                out.append((p[4:].strip() if pn else p, pn))
        return out
    last = None
    for c in re.split(r'[,.;]', body):
        c = re.sub(r'^\s*and\s+', '', c).strip()
        if not c:
            continue
        m = re.match(r'(.*?)' + verb + r'(.*)$', c)
        if m:
            lhs, rhs = m.groups()
            last = ('or' if re.search(r'\bor\b', lhs) else 'and', lits_of(lhs))
        elif last:
            rhs = c
        else:
            continue
        for eff, neg in lits_of(rhs):
            eqs[eff] = (last[0], last[1], neg)
    return eqs, priors, obs


def parse_cf_question(q):
    q = q.strip()
    m = re.match(r'(?i)would (\w+) occur if (not\s+)?(\w+) instead of (?:not\s+)?(\w+)\??', q)
    if m:
        return m.group(1), {m.group(3): 0 if m.group(2) else 1}
    m = re.match(r'(?i)assume (.*?),? and based on this assumption, further suppose (.*?)\. would (\w+) occur\??', q)
    if m:
        do = {}
        for part in (m.group(1), m.group(2)):
            for neg, v in re.findall(r'(not\s+)?\b([A-Z]\w*)', part):
                do[v] = 0 if neg else 1
        return m.group(3), do
    m = re.match(r'(?i)would (\w+) occur if (.*?)\??$', q)
    if m:
        do = {}
        for neg, v in re.findall(r'(not\s+)?\b([A-Z]\w*)', m.group(2)):
            do[v] = 0 if neg else 1
        return m.group(1), do
    raise ValueError('unparsed counterfactual question')


class CausalModel:
    def __init__(self, eqs, priors):
        self.eqs, self.priors = eqs, priors
        allv = set(eqs) | {p for e in eqs.values() for p, _ in e[1]}
        self.order = self._topo(allv)
        self.roots = [v for v in self.order if v not in eqs]

    def _topo(self, allv):
        order, seen = [], set()
        def visit(v):
            if v in seen:
                return
            seen.add(v)
            if v in self.eqs:
                for p, _ in self.eqs[v][1]:
                    visit(p)
            order.append(v)
        for v in sorted(allv):
            visit(v)
        return order

    def evaluate(self, exo, do):
        """Forward propagation of the structural equations (topological
        order; a few sweeps settle the rare cyclic descriptions)."""
        val = {v: 0 for v in self.order}
        val.update(exo)
        for _ in range(3):
            for v in self.order:
                if v in do:
                    val[v] = do[v]
                elif v in self.eqs:
                    op, lits, neg = self.eqs[v]
                    xs = [val.get(p, 0) ^ pn for p, pn in lits]
                    r = all(xs) if op == 'and' else any(xs)
                    val[v] = int(r) ^ int(neg)
        return val

    def counterfactual(self, target, do, evidence):
        """Abduction-action-prediction: enumerate exogenous assignments (DFS
        over a binary tree), keep those consistent with the evidence, apply
        the intervention, and return P(target)."""
        for ev in (evidence, {k: v for k, v in evidence.items() if k in self.roots}, {}):
            num = den = 0.0
            for bits in itertools.product((0, 1), repeat=len(self.roots)):
                exo = dict(zip(self.roots, bits))
                w = 1.0
                for r, b in exo.items():
                    p = self.priors.get(r, 0.5)
                    w *= p if b else 1 - p
                if w == 0:
                    continue
                if ev:
                    fact = self.evaluate(exo, {})
                    if any(fact.get(k) != v for k, v in ev.items()):
                        continue
                cf = self.evaluate(exo, do)
                den += w
                num += w * cf.get(target, 0)
            if den > 0:
                return num / den
        return 0.0


def parse_scm_logic(context):
    """Structural equations written as propositional logic:
       A → B,  (A ∧ ¬B) → C,  (A ∨ B) → ¬C,  C ↔ (A ∧ B);  bare literals are
       observations;  'X ~ Bern(p)' gives priors."""
    priors = {v: float(p) for v, p in re.findall(r'([A-Za-z]\w*)\s*~\s*Bern\(\s*([\d.]+)\s*\)', context)}
    text = re.sub(r'[A-Za-z]\w*\s*~\s*Bern\(\s*[\d.]+\s*\)', '', context)
    eqs, obs = {}, {}
    def lits(f):
        if f[0] in ('and', 'or'):
            out = [_literal(g) for g in f[1:]]
            return (f[0], out) if all(out) else None
        l = _literal(f)
        return ('and', [l]) if l else None
    formulas = []
    for line in _premise_lines(text):
        for piece in split_logic_line(line) or [line]:
            try:
                formulas.append(parse_logic(piece))
            except ParseError:
                pass
    for f in formulas:
        f, _ = _strip_forall(f)
        if f[0] in ('imp', 'iff'):
            body, head = (f[1], f[2])
            if _literal(body) and not _literal(head) and f[0] == 'iff':
                body, head = head, body
            hl, bl = _literal(head), lits(body)
            if hl and bl:
                eqs[hl[1]] = (bl[0], [(l[1], not l[0]) for l in bl[1]], not hl[0])
        else:
            for l in (_conj_literals(f) or []):
                obs[l[1]] = 1 if l[0] else 0
    return eqs, priors, obs


def answer_counterfactual(context, statement):
    if looks_like_logic(context) and not re.search(r'\bcauses?\b', context):
        eqs, priors, obs = parse_scm_logic(context)
    else:
        eqs, priors, obs = parse_scm(context)
    target, do = parse_cf_question(statement)
    m = CausalModel(eqs, priors)
    p = m.counterfactual(target, do, obs)
    return p > 0.5, {'equations': eqs, 'do': do, 'evidence': obs, 'P(target)': p}

## 8. Symbol normalisation
Merges symbols that are the same name spelt differently (case, plural, one-letter typo in a long name) and splits compound unary predicates such as `InMyHouse(x)` into `In(x, myHouse)` when that binary predicate exists.

In [ ]:
# -----------------------------------------------------------------------------
# 8. Symbol normalisation (representation clean-up before reasoning)
#    Hand-written logic often spells one symbol in several ways
#    (ResidentOf / Residentof, RareDisease / RareDiseases, brianWinter /
#    brianwinter, homelessCharity / homelessCharify).  Symbols of the same
#    kind and arity are merged when they differ only by case/punctuation,
#    a plural ending, or a single edit in a long name.
# -----------------------------------------------------------------------------
def _canon(n):
    return re.sub(r'[^a-z0-9]', '', n.lower())


def _lev1(a, b, k):
    """Levenshtein distance <= k (small k, banded DP)."""
    if abs(len(a) - len(b)) > k:
        return False
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (a[i - 1] != b[j - 1]))
        if min(cur) > k:
            return False
        prev = cur
    return prev[-1] <= k


_NEG_PREFIX = ('non', 'un', 'in', 'im', 'il', 'ir', 'dis', 'not', 'anti', 'mis')


def _similar(a, b):
    ca, cb = _canon(a), _canon(b)
    if ca == cb:
        return True
    if not ca or not cb or ca[0] != cb[0]:
        return False
    for x, y in ((ca, cb), (cb, ca)):
        if x == y + 's' or x == y + 'es' or (x.endswith('ies') and x[:-3] + 'y' == y):
            return True
        for p in _NEG_PREFIX:
            if x == p + y:
                return False
    if min(len(ca), len(cb)) >= 7 and _lev1(ca, cb, 1):
        return True
    if min(len(ca), len(cb)) >= 12 and _lev1(ca, cb, 2):
        return True
    return False


def _collect(f, preds, consts):
    def ct(t):
        if isinstance(t, tuple):
            for a in t[1:]:
                ct(a)
        elif not is_var(t):
            consts[t] += 1
    op = f[0]
    if op == 'atom':
        preds[(f[1], len(f[2]))] += 1
        for a in f[2]:
            ct(a)
    elif op == 'not':
        _collect(f[1], preds, consts)
    elif op in ('and', 'or', 'imp', 'iff', 'xor'):
        for g in f[1:]:
            _collect(g, preds, consts)
    elif op in ('forall', 'exists'):
        _collect(f[2], preds, consts)


def _merge_map(counter, key=lambda s: s, arity=lambda s: 0):
    syms = sorted(counter, key=lambda s: (-counter[s], str(s)))
    rep = {}
    for i, s in enumerate(syms):
        if s in rep:
            continue
        rep[s] = s
        for t in syms[i + 1:]:
            if t not in rep and arity(s) == arity(t) and _similar(key(s), key(t)):
                rep[t] = s
    return rep


def normalize_symbols(formulas):
    preds, consts = collections.Counter(), collections.Counter()
    for f in formulas:
        _collect(f, preds, consts)
    if '=' in {p for p, _ in preds}:
        preds = collections.Counter({k: v for k, v in preds.items() if k[0] != '='})
    pmap = _merge_map(preds, key=lambda s: s[0], arity=lambda s: s[1])
    cmap = _merge_map(consts)
    # split compound unary predicates:  InMyHouse(jojo) -> In(jojo, myHouse)
    split = {}
    binaries = [p for p, a in preds if a == 2]
    for (p, a) in preds:
        if a != 1 or preds[(p, a)] > 1:
            continue
        for q in binaries:
            if p != q and _canon(p).startswith(_canon(q)):
                rest = _canon(p)[len(_canon(q)):]
                for c in consts:
                    if _canon(c) == rest:
                        split[p] = (q, cmap.get(c, c))
    def ct(t):
        if isinstance(t, tuple):
            return (t[0],) + tuple(ct(a) for a in t[1:])
        return t if is_var(t) else cmap.get(t, t)
    def fa(at):
        name, args = at[1], tuple(ct(a) for a in at[2])
        if name in split and len(args) == 1:
            q, c = split[name]
            return ('atom', pmap.get((q, 2), (q, 2))[0], (args[0], c))
        if name == '=':
            return ('atom', '=', args)
        return ('atom', pmap.get((name, len(args)), (name,))[0], args)
    return [map_atoms(f, fa) for f in formulas]

## 9. The reasoning agent (engine selection)
Counterfactual questions → engine C. Rule-shaped knowledge bases with a ground literal query and only unary/binary predicates → engine A (closed world). Everything else → engine B (open world).

In [ ]:
# -----------------------------------------------------------------------------
# 9. The agent
# -----------------------------------------------------------------------------
class ReasoningAgent:
    def __init__(self, time_limit=0.15, closed_world='auto', normalize=True,
                 free_premise='forall', free_goal='exists'):
        self.prover_cfg = dict(time_limit=time_limit)
        self.closed_world = closed_world
        self.normalize = normalize
        self.free_premise, self.free_goal = free_premise, free_goal
        self.stats = collections.Counter()
        self._fc_cache = {}

    # ---- engine selection ---------------------------------------------------
    def _rule_kb(self, kb_key, kb):
        if kb_key in self._fc_cache:
            return self._fc_cache[kb_key]
        r = to_rules(kb)
        fc = ForwardChainer(*r) if r is not None else None
        if len(self._fc_cache) > 2000:
            self._fc_cache.clear()
        self._fc_cache[kb_key] = fc
        return fc

    def _use_cwa(self, kb, fc):
        if self.closed_world in (True, False):
            return self.closed_world
        if fc is None:
            return False
        # closed-world rule bases (RuleTaker-like) use unary/binary predicates
        # and no equality; anything richer is treated as open-world FOL
        preds, consts = collections.Counter(), collections.Counter()
        for f in kb:
            _collect(f, preds, consts)
        return all(p != '=' and a <= 2 for p, a in preds)

    def predict(self, context, statement, explain=False):
        self.stats['queries'] += 1
        if is_counterfactual(statement):
            self.stats['engine_scm'] += 1
            try:
                ans, info = answer_counterfactual(context, statement)
            except Exception:
                self.stats['scm_fail'] += 1
                return (False, ['could not parse causal model']) if explain else False
            if explain:
                return ans, ['structural equations: %s' % info['equations'],
                             'intervention do(%s), evidence %s' % (info['do'], info['evidence']),
                             'P(%s) = %.3f' % (statement, info['P(target)'])]
            return ans
        kb_key = context if isinstance(context, str) else tuple(context)
        kb = parse_kb(context, self.stats, self.free_premise)
        goal = parse_statement(statement, self.stats, self.free_goal)
        if not goal:
            self.stats['goal_fail'] += 1
            return (False, ['query not parsed']) if explain else False
        goal = goal[0] if len(goal) == 1 else flat('and', goal)
        if self.normalize and looks_like_logic(str(context)):
            *kb, goal = normalize_symbols(kb + [goal])
            kb_key = (kb_key, statement)
        fc = self._rule_kb(kb_key, kb)
        gl = _literal(_strip_forall(goal)[0]) if goal else None
        ground_goal = gl is not None and not any(is_var(a) or isinstance(a, tuple) for a in gl[2])
        if fc is not None and ground_goal and self._use_cwa(kb, fc):
            self.stats['engine_forward'] += 1
            ans = fc.holds(*gl)
            if explain:
                return ans, fc.explain(*gl)
            return ans
        self.stats['engine_resolution'] += 1
        ans, lines = self._resolve(kb, goal)
        return (ans, lines) if explain else ans

    def _resolve(self, kb, goal):
        try:
            kbc = []
            for f in kb:
                kbc.extend(to_clauses(f))
            gc = to_clauses(('not', goal))
        except (OverflowError, RecursionError):
            self.stats['cnf_overflow'] += 1
            return False, []
        kbc += equality_axioms(kbc + gc)
        prover = ResolutionProver(**self.prover_cfg)
        ok = prover.prove(kbc, gc)
        return ok, prover.proof_lines() if ok else []

## 10. Public datasets
RuleTaker (`tasksource/ruletaker`), CounterBench (`CounterBench/CounterBench`) and FOLIO (`tasksource/folio`, the
public mirror of `yale-nlp/FOLIO`) are downloaded from Hugging Face. Labels are mapped to the binary entailment flag:
RuleTaker `entailment` → True; FOLIO `True` → True (`False`/`Uncertain` → False); CounterBench `yes` → True.
FOLIO is given to the system as its first-order-logic annotation (`premises-FOL`, `conclusion-FOL`), i.e. the logic
statements are given directly.

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download

def fetch(repo, filename):
    return hf_hub_download(repo, filename, repo_type='dataset', local_dir=DATA_DIR)

def load_ruletaker(split='test', n=None, seed=0):
    files = {'dev': 'data/dev-00000-of-00001-480b21836eb3db96.parquet',
             'test': 'data/test-00000-of-00001-402c217dbcd02599.parquet'}
    df = pd.read_parquet(fetch('tasksource/ruletaker', files[split]))
    if n:
        df = df.sample(n=n, random_state=seed)
    return [dict(id='ruletaker_%s_%d' % (split, i), context=r.context, statement=r.question,
                 label=r.label == 'entailment', group='RuleTaker/' + r.config)
            for i, r in zip(df.index, df.itertuples())]

def load_folio(splits=('train', 'validation')):
    out = []
    for sp in splits:
        with open(fetch('tasksource/folio', 'folio_v2_%s.jsonl' % sp), encoding='utf8') as fh:
            for l in fh:
                x = json.loads(l)
                out.append(dict(id='folio_%s_%s' % (sp, x['example_id']), context=x['premises-FOL'],
                                statement=x['conclusion-FOL'], label=x['label'] == 'True',
                                group='FOLIO/' + sp, gold3=x['label']))
    return out

def load_counterbench():
    out = []
    for f in ['data_balanced_alpha_V1.json', 'data_balanced_backdoor_V2.json']:
        with open(fetch('CounterBench/CounterBench', f), encoding='utf8') as fh:
            for x in json.load(fh):
                out.append(dict(id='counterbench_%s_%s' % (f.split('_')[2], x['question_id']),
                                context=x['given_info'], statement=x['question'],
                                label=x['answer'] == 'yes', group='CounterBench/' + x['type']))
    return out

ruletaker = load_ruletaker('test', RULETAKER_EVAL)
folio = load_folio()
counterbench = load_counterbench()
print('RuleTaker test:', len(ruletaker), '| FOLIO:', len(folio), '| CounterBench:', len(counterbench))
print(ruletaker[0]['context'][:300], '...\nQ:', ruletaker[0]['statement'], ruletaker[0]['label'])
print(folio[0]['context'], '\nQ:', folio[0]['statement'], folio[0]['gold3'])
print(counterbench[0]['context'], '\nQ:', counterbench[0]['statement'], counterbench[0]['label'])

C:\Users\91880\AppData\Local\Temp\claude\E--Assignment-AI\9c622fa6-6ea1-4694-b9b7-e3647ba2577f\scratchpad\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RuleTaker test: 151911 | FOLIO: 1204 | CounterBench: 1200
Anne is big. Anne is cold. Anne is furry. Anne is green. Anne is red. Anne is rough. Erin is cold. Erin is furry. Erin is green. Erin is round. If something is cold then it is furry. If something is round then it is rough. If something is green and cold then it is furry. All furry things are round.  ...
Q: Anne is red. True
∀x (DrinkRegularly(x, coffee) → IsDependentOn(x, caffeine))
∀x (DrinkRegularly(x, coffee)  ∨ (¬WantToBeAddictedTo(x, caffeine)))
∀x (¬WantToBeAddictedTo(x, caffeine) → ¬AwareThatDrug(x, caffeine))
¬(Student(rina) ⊕  ¬AwareThatDrug(rina, caffeine))
¬(IsDependentOn(rina, caffeine) ⊕ Student(rina)) 
Q: ¬WantToBeAddictedTo(rina, caffeine) ∨ (¬AwareThatDrug(rina, caffeine)) True
We know that Ziklo causes Blaf, Blaf causes Trune, Trune causes Vork, and Vork causes Lumbo. 
Q: Would Lumbo occur if not Ziklo instead of Ziklo? False


## 11. Evaluation on the public datasets
Accuracy = correct predictions / queries × 100; inference time is the wall-clock time of the reasoning system (parsing + inference).

In [ ]:
def evaluate(items, agent=None, name=''):
    agent = agent or ReasoningAgent()
    acc, tot = collections.Counter(), collections.Counter()
    preds = []
    t0 = time.perf_counter()
    for x in items:
        p = agent.predict(x['context'], x['statement'])
        preds.append(p)
        for k in (x['group'].split('/')[0], x['group']):
            tot[k] += 1
            acc[k] += (p == x['label'])
    dt = time.perf_counter() - t0
    rows = [(k, tot[k], 100.0 * acc[k] / tot[k]) for k in sorted(tot)]
    return preds, dt, pd.DataFrame(rows, columns=['subset', 'queries', 'accuracy_%']), agent

results = {}
for name, items in [('RuleTaker', ruletaker), ('FOLIO', folio), ('CounterBench', counterbench)]:
    preds, dt, table, agent = evaluate(items)
    results[name] = dict(preds=preds, time=dt, table=table, n=len(items),
                         acc=100.0 * sum(p == x['label'] for p, x in zip(preds, items)) / len(items))
    print('=== %s: accuracy %.2f%%  (%d queries, inference time %.1fs, %.2f ms/query)' % (
        name, results[name]['acc'], len(items), dt, 1000 * dt / len(items)))
    print(table.to_string(index=False))
    print('engines used:', {k: v for k, v in agent.stats.items() if k.startswith('engine')})
    print()

=== RuleTaker: accuracy 99.97%  (151911 queries, inference time 71.7s, 0.47 ms/query)
                      subset  queries  accuracy_%
                   RuleTaker   151911   99.973011
           RuleTaker/NatLang     7793  100.000000
           RuleTaker/depth-0    17347  100.000000
           RuleTaker/depth-1    17788  100.000000
           RuleTaker/depth-2    17741   99.988727
           RuleTaker/depth-3    17982   99.955511
        RuleTaker/depth-3ext    23222   99.965550
RuleTaker/depth-3ext-NatLang    31017   99.974208
           RuleTaker/depth-5    19021   99.921140
engines used: {'engine_forward': 151911}



=== FOLIO: accuracy 82.72%  (1204 queries, inference time 37.2s, 30.94 ms/query)
          subset  queries  accuracy_%
           FOLIO     1204   82.724252
     FOLIO/train     1001   81.818182
FOLIO/validation      203   87.192118
engines used: {'engine_resolution': 1089, 'engine_forward': 115}



=== CounterBench: accuracy 95.00%  (1200 queries, inference time 0.5s, 0.38 ms/query)
                  subset  queries  accuracy_%
            CounterBench     1200        95.0
   CounterBench/backdoor      200        90.0
      CounterBench/basic      250       100.0
CounterBench/conditional      250        92.0
      CounterBench/joint      250        96.0
     CounterBench/nested      250        96.0
engines used: {'engine_scm': 1200}



In [ ]:
total_n = sum(r['n'] for r in results.values())
total_correct = sum(r['acc'] * r['n'] / 100 for r in results.values())
total_time = sum(r['time'] for r in results.values())
summary = pd.DataFrame([(k, r['n'], round(r['acc'], 2), round(r['time'], 1), round(1000 * r['time'] / r['n'], 3))
                        for k, r in results.items()],
                       columns=['dataset', 'queries', 'accuracy_%', 'inference_time_s', 'ms_per_query'])
print(summary.to_string(index=False))
print()
print('FINAL ACCURACY (all public queries, micro-average): %.2f%%' % (100.0 * total_correct / total_n))
print('FINAL ACCURACY (mean of the three datasets)       : %.2f%%' % (sum(r['acc'] for r in results.values()) / 3))
print('TOTAL INFERENCE TIME: %.1f s for %d queries' % (total_time, total_n))

     dataset  queries  accuracy_%  inference_time_s  ms_per_query
   RuleTaker   151911       99.97              71.7         0.472
       FOLIO     1204       82.72              37.2        30.935
CounterBench     1200       95.00               0.5         0.375

FINAL ACCURACY (all public queries, micro-average): 99.80%
FINAL ACCURACY (mean of the three datasets)       : 92.57%
TOTAL INFERENCE TIME: 109.4 s for 154315 queries


## 12. Experiments
1. **Logic given directly for RuleTaker.** The English theories are converted to FOL text in three syntaxes
   (Unicode, ASCII, word operators) and fed back through the *logic* front-end.
2. **Closed vs open world on RuleTaker** (forward chaining with NAF vs. resolution).
3. **FOLIO ablations**: symbol normalisation, unique-names assumption, prover time budget.
4. **CounterBench** given as propositional logic (`(A ∧ ¬B) → C`) instead of English.

In [ ]:
def fol_text(f, style):
    S = {'unicode': ('¬', '∧', '→', '∀%s '), 'ascii': ('~', '&', '->', 'forall %s. '),
         'words': ('not ', 'and', 'implies', 'forall %s ')}[style]
    def go(f):
        op = f[0]
        if op == 'atom':
            return '%s(%s)' % (f[1][:1].upper() + f[1][1:], ', '.join(a.lstrip('?') for a in f[2]))
        if op == 'not':
            return S[0] + go(f[1])
        if op == 'and':
            return '(' + (' %s ' % S[1]).join(go(g) for g in f[1:]) + ')'
        if op == 'imp':
            return '(' + go(f[1]) + ' %s ' % S[2] + go(f[2]) + ')'
        if op == 'forall':
            return S[3] % f[1].lstrip('?') + go(f[2])
    return go(f)

if RUN_EXPERIMENTS:
    sample = random.Random(1).sample(ruletaker, min(5000, len(ruletaker)))
    exp_rows = []
    for style in ['unicode', 'ascii', 'words']:
        conv = [dict(x, context='\n'.join(fol_text(f, style) for f in parse_kb(x['context'])),
                     statement=fol_text(parse_statement(x['statement'])[0], style)) for x in sample]
        preds, dt, table, _ = evaluate(conv)
        a = 100.0 * sum(p == x['label'] for p, x in zip(preds, conv)) / len(conv)
        exp_rows.append(('RuleTaker as logic (%s)' % style, len(conv), round(a, 2), round(dt, 1)))
    print(conv[0]['context'][:400], '\nQ:', conv[0]['statement'])
    sub = sample[:1500]
    for cw in [True, False]:
        preds, dt, _, _ = evaluate(sub, ReasoningAgent(closed_world=cw))
        a = 100.0 * sum(p == x['label'] for p, x in zip(preds, sub)) / len(sub)
        exp_rows.append(('RuleTaker, %s' % ('closed world: forward chaining + NAF' if cw else 'open world: resolution'),
                         len(sub), round(a, 2), round(dt, 1)))
    print(pd.DataFrame(exp_rows, columns=['experiment', 'queries', 'accuracy_%', 'time_s']).to_string(index=False))

Eat(cat, tiger)
Nice(cat)
Need(cat, rabbit)
not Visit(cat, lion)
Eat(lion, cat)
Green(lion)
Kind(lion)
Young(lion)
Need(lion, rabbit)
Need(lion, tiger)
Eat(rabbit, cat)
not Eat(rabbit, tiger)
Nice(rabbit)
Young(rabbit)
Eat(tiger, rabbit)
Young(tiger)
forall x (Nice(x) implies Need(x, tiger))
forall x (Visit(x, tiger) implies Eat(tiger, lion))
forall x ((Blue(x) and Need(x, cat)) implies Eat(cat, t 
Q: not Need(cat, cat)


                                     experiment  queries  accuracy_%  time_s
                   RuleTaker as logic (unicode)     5000      100.00    17.8
                     RuleTaker as logic (ascii)     5000      100.00    13.6
                     RuleTaker as logic (words)     5000      100.00     5.8
RuleTaker, closed world: forward chaining + NAF     1500      100.00     0.8
              RuleTaker, open world: resolution     1500       74.27     3.0


In [ ]:
if RUN_EXPERIMENTS:
    def folio_acc(agent):
        t0 = time.perf_counter()
        ok = sum(agent.predict(x['context'], x['statement']) == x['label'] for x in folio)
        return round(100.0 * ok / len(folio), 2), round(time.perf_counter() - t0, 1)

    class NoUNA(ReasoningAgent):
        def _resolve(self, kb, goal):
            global equality_axioms
            saved = equality_axioms
            def no_una(clauses):
                return [c for c in saved(clauses) if not (len(c) == 1 and not c[0][0] and c[0][1] == '='
                        and not any(is_var(a) for a in c[0][2]))]
            equality_axioms = no_una
            try:
                return ReasoningAgent._resolve(self, kb, goal)
            finally:
                equality_axioms = saved

    rows = []
    for name, agent in [('resolution only, raw symbols, no UNA', NoUNA(normalize=False, closed_world=False)),
                        ('+ symbol normalisation', NoUNA(closed_world=False)),
                        ('+ unique-names assumption', ReasoningAgent(closed_world=False)),
                        ('+ CWA engine for rule-shaped KBs (final)', ReasoningAgent()),
                        ('final, prover budget 0.05 s', ReasoningAgent(time_limit=0.05)),
                        ('final, prover budget 0.5 s', ReasoningAgent(time_limit=0.5))]:
        rows.append((name,) + folio_acc(agent))
    folio_exp = pd.DataFrame(rows, columns=['FOLIO configuration', 'accuracy_%', 'time_s'])
    print(folio_exp.to_string(index=False))
    gold3 = collections.Counter((x['gold3'], p) for x, p in zip(folio, results['FOLIO']['preds']))
    print('\nFOLIO confusion (gold 3-way label vs predicted flag):')
    print(pd.DataFrame([[gold3[(g, True)], gold3[(g, False)]] for g in ['True', 'False', 'Uncertain']],
                       index=['True', 'False', 'Uncertain'], columns=['pred True', 'pred False']))

                     FOLIO configuration  accuracy_%  time_s
    resolution only, raw symbols, no UNA       79.82    34.5
                  + symbol normalisation       82.81    33.4
               + unique-names assumption       82.89    33.6
+ CWA engine for rule-shaped KBs (final)       82.72    33.0
             final, prover budget 0.05 s       82.64    13.6
              final, prover budget 0.5 s       82.81    96.4

FOLIO confusion (gold 3-way label vs predicted flag):
           pred True  pred False
True             297         163
False             29         322
Uncertain         16         377


In [ ]:
if RUN_EXPERIMENTS:
    conv = []
    for x in counterbench:
        eqs, priors, obs = parse_scm(x['context'])
        lines = ['(%s) → %s%s' % ((' ∧ ' if op == 'and' else ' ∨ ').join(('¬' if n else '') + p for p, n in lits),
                                  '¬' if neg else '', v) for v, (op, lits, neg) in eqs.items()]
        lines += ['%s ~ Bern(%s)' % (v, p) for v, p in priors.items()]
        lines += [('' if b else '¬') + v for v, b in obs.items()]
        conv.append(dict(x, context='\n'.join(lines)))
    preds, dt, table, _ = evaluate(conv)
    print(conv[1]['context'], '\nQ:', conv[1]['statement'])
    print('CounterBench given as logic: accuracy %.2f%%' % (100.0 * sum(p == x['label'] for p, x in zip(preds, conv)) / len(conv)))

(Glent) → Razz
(Razz) → Pex
(Pex) → Zurn
(Zurn) → Zlim 
Q: Would Zlim occur if not Glent instead of Glent?
CounterBench given as logic: accuracy 95.00%


## 13. Reasoning traces
The agent can return the derivation it used: the forward-chaining proof tree, the resolution refutation, or the causal computation.

In [ ]:
agent = ReasoningAgent()
demo = [
    ('Bob is young. Bob is kind. If someone is young then they are smart. '
     'If someone is smart and kind then they are successful.', 'Bob is successful.'),
    ('Young(bob)\nKind(bob)\n∀x (Young(x) → Smart(x))\n∀x (Smart(x) ∧ Kind(x) → Successful(x))', 'Successful(bob)'),
    ('Anne is kind. Kind things are blue. If something is blue and not red then it is cold.', 'Anne is cold.'),
]
ex_rt = next(x for x in ruletaker if x['group'].endswith('depth-3') and x['label']
             and len(agent.predict(x['context'], x['statement'], explain=True)[1]) >= 6)
demo.append((ex_rt['context'], ex_rt['statement']))
ex_fo = next(x for x, p in zip(folio, results['FOLIO']['preds']) if p and x['context'].count('\n') >= 4)
demo.append((ex_fo['context'], ex_fo['statement']))
demo.append((counterbench[3]['context'], counterbench[3]['statement']))
for ctx, q in demo:
    ans, trace = agent.predict(ctx, q, explain=True)
    print('CONTEXT:', ctx if len(ctx) < 700 else ctx[:700] + ' ...')
    print('QUERY  :', q, '  =>', ans)
    print('\n'.join(trace))
    print('-' * 100)

CONTEXT: Bob is young. Bob is kind. If someone is young then they are smart. If someone is smart and kind then they are successful.
QUERY  : Bob is successful.   => True
successful(bob)   [rule: smart(x) ∧ kind(x) → successful(x)]
  smart(bob)   [rule: young(x) → smart(x)]
    young(bob)   [fact]
  kind(bob)   [fact]
----------------------------------------------------------------------------------------------------
CONTEXT: Young(bob)
Kind(bob)
∀x (Young(x) → Smart(x))
∀x (Smart(x) ∧ Kind(x) → Successful(x))
QUERY  : Successful(bob)   => True
Successful(bob)   [rule: Smart(x) ∧ Kind(x) → Successful(x)]
  Smart(bob)   [rule: Young(x) → Smart(x)]
    Young(bob)   [fact]
  Kind(bob)   [fact]
----------------------------------------------------------------------------------------------------
CONTEXT: Anne is kind. Kind things are blue. If something is blue and not red then it is cold.
QUERY  : Anne is cold.   => True
cold(anne)   [rule: blue(x) ∧ not red(x) → cold(x)]
  blue(anne)   [rule

## 14. Predictions for the test set → `submission.csv`

In [ ]:
def read_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == '.csv':
        return pd.read_csv(path)
    if ext in ('.tsv', '.txt'):
        return pd.read_csv(path, sep='\t')
    if ext == '.jsonl':
        return pd.read_json(path, lines=True)
    if ext == '.json':
        try:
            return pd.read_json(path)
        except ValueError:
            return pd.read_json(path, lines=True)
    if ext == '.parquet':
        return pd.read_parquet(path)
    raise ValueError('unsupported test file ' + path)

def pick(df, names):
    for n in names:
        for c in df.columns:
            if c.lower() == n:
                return c
    return None

def to_text(v):
    if isinstance(v, (list, tuple)):
        return '\n'.join(map(str, v))
    return '' if v is None or (isinstance(v, float) and v != v) else str(v)

def predict_file(path):
    df = read_any(path)
    cid = pick(df, ['id', 'example_id', 'question_id', 'qid', 'idx', 'index'])
    cctx = pick(df, ['context', 'premises-fol', 'premises_fol', 'premises', 'theory', 'given_info',
                     'knowledge_base', 'kb', 'facts', 'logic_context'])
    cq = pick(df, ['statement', 'conclusion-fol', 'conclusion_fol', 'conclusion', 'question', 'query',
                   'hypothesis', 'logic_statement'])
    ids = df[cid].tolist() if cid else list(range(len(df)))
    agent = ReasoningAgent()
    t0 = time.perf_counter()
    preds = [agent.predict(to_text(c), to_text(q)) for c, q in zip(df[cctx], df[cq])]
    dt = time.perf_counter() - t0
    out = pd.DataFrame({'id': ids, LABEL_COLUMN: preds})
    gold = pick(df, ['flag', 'label', 'answer', 'gold'])
    if gold:
        g = df[gold].map(lambda v: str(v).strip().lower() in ('true', '1', 'yes', 'entailment'))
        print('accuracy on %s: %.2f%%' % (path, 100.0 * (g.values == out[LABEL_COLUMN].values).mean()))
    print('inference time: %.1fs for %d queries' % (dt, len(df)))
    return out

if os.path.exists(TEST_PATH):
    submission = predict_file(TEST_PATH)
else:
    print('No private test file at %r: writing predictions for the public evaluation sets.' % TEST_PATH)
    rows = []
    for name, items in [('RuleTaker', ruletaker), ('FOLIO', folio), ('CounterBench', counterbench)]:
        rows += [(x['id'], p) for x, p in zip(items, results[name]['preds'])]
    submission = pd.DataFrame(rows, columns=['id', LABEL_COLUMN])
submission.to_csv(OUTPUT_PATH, index=False)
print('wrote', OUTPUT_PATH, submission.shape)
print(submission.head())

No private test file at 'test.csv': writing predictions for the public evaluation sets.


wrote submission.csv (154315, 2)
                 id   flag
0  ruletaker_test_0   True
1  ruletaker_test_1  False
2  ruletaker_test_2   True
3  ruletaker_test_3  False
4  ruletaker_test_4   True
